# Portfolio Manager + Gonçalo's Macro/Risk – integrated graph

**What this notebook does:** connects Francesco's part (weights → orders) to Gonçalo's LangGraph skeleton (Macro/News + Risk rules).

```
START → macro (Gonçalo) → portfolio_manager (Francesco, placeholder for now) → orders (Francesco) → risk (Gonçalo) → END
```

- **macro**: Google News headlines + Gemini summary (Gonçalo's code)
- **portfolio_manager**: proposes **target weights** (for now fixed weights; the AI version comes next)
- **orders**: `weights_to_orders` turns the weights into max 5 feasible orders, in Gonçalo's order format
- **risk**: `check_risk_rules` (Gonçalo) does the final check on our risk limits

Changes vs. Gonçalo's notebook: starting capital €10,000; one shared minimum order (€100, to agree); works on Colab (key from Secrets) and locally (key from `.env`); adds the `leverage` column if `prices.csv` doesn't have it yet.

## 0. Install (once)

In [ ]:
%pip install -q pandas feedparser langgraph langchain-openai openai python-dotenv

## 1. Settings (competition rules + our risk rules)

In [ ]:
import os, json, math, time
from pathlib import Path
import pandas as pd

# Find the project folder (the one with data/prices.csv); on Colab, prices.csv can also be uploaded at the top level
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "data" / "prices.csv").exists():
        os.chdir(folder); break
PRICES_FILE = Path("data/prices.csv") if Path("data/prices.csv").exists() else Path("prices.csv")
print("Using prices file:", PRICES_FILE.resolve())

# ---- Competition rules (QTEM FAQ) - do not change ----
MAX_ORDERS_PER_DAY = 5
MIN_CASH_EUR       = 0.0
STARTING_CAPITAL   = 10_000.0          # Belfius slides: €10,000 per team

# ---- Our risk rules (team decision, to be confirmed) ----
MAX_WEIGHT_PER_ETF   = 0.20            # max 20% in one ETF
MAX_LEVERAGED_WEIGHT = 0.15            # max 15% in all leveraged/inverse ETFs together
MIN_ORDER_EUR        = 100.0           # one shared minimum for both functions (to agree)
DECIMALS             = 4               # fractional units allowed

# Leverage of each leveraged/inverse ETF (same values as code/download_etfs_revised.py)
LEVERAGE = {"NASDAQ 3X": 3, "COPPER LVG": 3, "PALLADIUM LVG": 2,
            "EURO STOXX 2X": 2, "S&P 500 SHORT": -1, "STOXX SHORT": -2}

## 2. Load data
If `prices.csv` is the old version (no `leverage` column), the column is added here from the table above.

In [ ]:
prices = pd.read_csv(PRICES_FILE, parse_dates=["date"])
if "leverage" not in prices.columns:
    print("Old prices.csv (no leverage column): adding it from the LEVERAGE table")
    prices["leverage"] = prices["code"].map(LEVERAGE).fillna(1).astype(int)

universe   = prices.groupby("code").agg(theme=("theme", "last"), leverage=("leverage", "last")).reset_index()
last_close = prices.sort_values("date").groupby("code")["close"].last()   # T-1 raw close
PRICES     = last_close.to_dict()

print(f"{len(universe)} ETFs, prices up to {prices['date'].max():%Y-%m-%d}")
print("Leveraged/inverse:", ", ".join(universe.loc[universe["leverage"] != 1, "code"]))

## 3. Portfolio format
One single format for everyone: `{"cash": 10000.0, "units": {"WORLD": 15.2, ...}}`.
`to_goncalo_portfolio` converts it to the table Gonçalo's `check_risk_rules` expects (columns `code`, `units` + a `CASH` row).

In [ ]:
def new_portfolio():
    return {"cash": STARTING_CAPITAL, "units": {}}

def to_goncalo_portfolio(pf):
    rows = [{"code": "CASH", "units": pf["cash"]}] + [{"code": c, "units": u} for c, u in pf["units"].items()]
    return pd.DataFrame(rows)

def portfolio_value(pf, prices=PRICES):
    return pf["cash"] + sum(u * prices[c] for c, u in pf["units"].items())

## 4. Francesco – `weights_to_orders` (adapted to Gonçalo's order format)
Output orders now look like Gonçalo's: `{"code": "WORLD", "action": "buy", "amount_eur": 2000.0}` (+ `units` and `price_T-1` for information).

In [ ]:
def check_weights(target_weights, prices=PRICES):
    errors = []
    for c, w in target_weights.items():
        if c not in prices: errors.append(f"{c}: not in the ETF list")
        if w < 0:           errors.append(f"{c}: negative weight ({w})")
    if sum(target_weights.values()) > 1.0001:
        errors.append(f"weights sum to {sum(target_weights.values()):.4f} > 1 (no borrowing)")
    return errors


def weights_to_orders(pf, target_weights, prices=PRICES,
                      max_orders=MAX_ORDERS_PER_DAY, min_trade_eur=MIN_ORDER_EUR):
    """Target weights -> at most 5 feasible orders (sells first, biggest moves first, every buy funded)."""
    errors = check_weights(target_weights, prices)
    if errors:
        raise ValueError("Invalid weights:\n- " + "\n- ".join(errors))

    total = portfolio_value(pf, prices)

    # 1. Gap between current and target value for every ETF (ignore tiny ones)
    gaps = []
    for c in set(pf["units"]) | set(target_weights):
        gap = target_weights.get(c, 0.0) * total - pf["units"].get(c, 0.0) * prices[c]
        if abs(gap) >= min_trade_eur:
            gaps.append((c, gap))
    gaps.sort(key=lambda x: abs(x[1]), reverse=True)          # biggest moves first
    sells_left = [g for g in gaps if g[1] < 0]

    # 2. Pick max 5 moves; a buy is kept only if it can be paid (pull in a sell if needed)
    chosen, cash_avail, not_executed = [], pf["cash"], []
    for c, gap in gaps:
        if (c, gap) in chosen:
            continue
        if len(chosen) >= max_orders:
            not_executed.append((c, round(gap, 2))); continue
        if gap < 0:
            chosen.append((c, gap)); sells_left.remove((c, gap)); cash_avail += -gap; continue
        while cash_avail < gap and sells_left and len(chosen) < max_orders - 1:
            s = sells_left.pop(0); chosen.append(s); cash_avail += -s[1]
        amount = min(gap, cash_avail)
        if amount < min_trade_eur:
            not_executed.append((c, round(gap, 2))); continue
        chosen.append((c, amount)); cash_avail -= amount

    # 3. Build orders in Gonçalo's format: sells first, then buys (biggest first)
    orders, cash = [], pf["cash"]
    for c, gap in chosen:
        if gap < 0:
            held = pf["units"].get(c, 0.0)
            units = held if target_weights.get(c, 0.0) == 0 else min(held, round(-gap / prices[c], DECIMALS))
            cash += units * prices[c]
            orders.append({"code": c, "action": "sell", "amount_eur": round(units * prices[c], 2),
                           "units": units, "price_T-1": round(prices[c], 4)})
    for c, amount in sorted([x for x in chosen if x[1] > 0], key=lambda x: x[1], reverse=True):
        amount = min(amount, cash)
        units = math.floor(amount / prices[c] * 10**DECIMALS) / 10**DECIMALS
        cash -= units * prices[c]
        orders.append({"code": c, "action": "buy", "amount_eur": round(units * prices[c], 2),
                       "units": units, "price_T-1": round(prices[c], 4)})

    return {"orders": orders, "cash_after": round(cash, 2), "not_executed": not_executed}

## 5. Gonçalo – `check_risk_rules` (unchanged, only reads the shared settings above)

In [ ]:
def check_risk_rules(orders, portfolio):
    px = last_close
    lev = universe.set_index("code")["leverage"].to_dict()
    holdings = portfolio[portfolio["code"] != "CASH"].set_index("code")["units"].to_dict()
    cash = float(portfolio.loc[portfolio["code"] == "CASH", "units"].sum())
    value = lambda c: holdings.get(c, 0) * px.get(c, 0)
    total = cash + sum(value(c) for c in holdings)
    approved, rejected = [], []

    for o in orders[MAX_ORDERS_PER_DAY:]:
        rejected.append({**o, "reason": f"more than {MAX_ORDERS_PER_DAY} orders"})
    orders = orders[:MAX_ORDERS_PER_DAY]

    valid = []
    for o in orders:
        c, a, amt = o.get("code"), str(o.get("action", "")).lower(), float(o.get("amount_eur", 0))
        if c not in lev:                 rejected.append({**o, "reason": "ETF not in our list"})
        elif a not in ("buy", "sell"):   rejected.append({**o, "reason": "action must be buy or sell"})
        elif amt < MIN_ORDER_EUR:        rejected.append({**o, "reason": f"below {MIN_ORDER_EUR:.0f} EUR"})
        else:                            valid.append({"code": c, "action": a, "amount_eur": amt})

    for o in [o for o in valid if o["action"] == "sell"]:
        held = value(o["code"])
        if held <= 0:
            rejected.append({**o, "reason": "we don't hold it"}); continue
        amt = min(o["amount_eur"], held)
        holdings[o["code"]] -= amt / px[o["code"]]
        cash += amt
        approved.append({**o, "amount_eur": round(amt, 2), "units": round(amt / px[o["code"]], 4),
                         "price_T-1": round(px[o["code"]], 4), "note": "cut to what we hold" if amt < o["amount_eur"] else ""})

    lev_value = sum(value(c) for c in holdings if lev.get(c, 1) != 1)
    for o in [o for o in valid if o["action"] == "buy"]:
        c, amt, why = o["code"], o["amount_eur"], []
        room = MAX_WEIGHT_PER_ETF * total - value(c)
        if amt > room:
            amt = max(room, 0); why.append(f"max {MAX_WEIGHT_PER_ETF:.0%} per ETF")
        if lev.get(c, 1) != 1:
            room_lev = MAX_LEVERAGED_WEIGHT * total - lev_value
            if amt > room_lev:
                amt = max(room_lev, 0); why.append(f"max {MAX_LEVERAGED_WEIGHT:.0%} in leveraged ETFs")
        if amt > cash - MIN_CASH_EUR:
            amt = max(cash - MIN_CASH_EUR, 0); why.append("not enough cash")
        if amt < MIN_ORDER_EUR:
            rejected.append({**o, "reason": "; ".join(why) or "too small"}); continue
        holdings[c] = holdings.get(c, 0) + amt / px[c]
        cash -= amt
        if lev.get(c, 1) != 1: lev_value += amt
        approved.append({**o, "amount_eur": round(amt, 2), "units": round(amt / px[c], 4),
                         "price_T-1": round(px[c], 4), "note": "; ".join(why)})

    weights = {c: float(round(value(c) / total, 3)) for c in holdings if holdings[c] > 1e-9}
    weights["CASH"] = float(round(cash / total, 3))
    return {"approved": approved, "rejected": rejected, "cash_after_eur": round(cash, 2), "weights_after": weights}

## 6. Quick test without AI: weights → orders → risk check

In [ ]:
pf = new_portfolio()
test_weights = {"WORLD": 0.20, "S&P 500": 0.20, "GOLD": 0.15, "EUR GOVT BOND": 0.15,
                "NASDAQ 3X": 0.25}            # NASDAQ 3X breaks the 15% leveraged cap on purpose

res = weights_to_orders(pf, test_weights)
print("ORDERS FROM weights_to_orders:"); display(pd.DataFrame(res["orders"]))
risk = check_risk_rules(res["orders"], to_goncalo_portfolio(pf))
print("APPROVED BY RISK:");  display(pd.DataFrame(risk["approved"]))
print("REJECTED:", risk["rejected"])
print("Weights after:", risk["weights_after"])

## 7. Gonçalo – Macro/News
Headlines from Google News (free) + Gemini summary. The key is read from **Colab Secrets** (`LLM_API_KEY`) or, on a computer, from the `.env` file.
If no key or the model is busy, the graph still runs and shows the raw headlines.

In [ ]:
import feedparser
from urllib.parse import quote

NEWS_TOPICS = {
    "interest_rates": "ECB OR Federal Reserve interest rates",
    "ecb_europe":     "ECB Lagarde euro zone",
    "geopolitics":    "geopolitical risk markets",
    "commodities":    "oil OR copper OR gold prices",
}
N_HEADLINES = 8

def get_headlines(query, n=N_HEADLINES):
    url = f"https://news.google.com/rss/search?q={quote(query)}+when:7d&hl=en&gl=US&ceid=US:en"
    try:
        entries = feedparser.parse(url).entries[:n]
        return [f"- {e.title} ({e.get('published', '')[:16]})" for e in entries] or ["- no headlines found"]
    except Exception as e:
        return [f"- news unavailable ({type(e).__name__})"]

# ---- API key: Colab Secrets first, then .env ----
try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
except Exception:
    from dotenv import load_dotenv; load_dotenv()
os.environ.setdefault("LLM_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/")

CANDIDATE_MODELS = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-flash-latest"]
os.environ["LLM_MODEL"] = ""
if os.environ.get("LLM_API_KEY"):
    from openai import OpenAI
    client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"], timeout=20, max_retries=0)
    for m in CANDIDATE_MODELS:
        try:
            client.chat.completions.create(model=m, messages=[{"role": "user", "content": "Say hi"}])
            os.environ["LLM_MODEL"] = m; break
        except Exception as e:
            print(f"{m}: not available ({type(e).__name__})")
print("Gemini model:", os.environ["LLM_MODEL"] or "none (the graph will run without the AI summary)")

THEMES = sorted(universe["theme"].unique())

def get_llm():
    if not all(os.environ.get(k) for k in ("LLM_BASE_URL", "LLM_MODEL", "LLM_API_KEY")):
        return None
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"],
                      model=os.environ["LLM_MODEL"], temperature=0, timeout=90, max_retries=3)

def get_macro_news(retries=2, pause=20):
    news = {t: get_headlines(q) for t, q in NEWS_TOPICS.items()}
    news_text = "\n\n".join(f"[{t}]\n" + "\n".join(h) for t, h in news.items())
    llm = get_llm()
    if llm is None:
        return {"summary": "[AI not configured]", "headlines": news}
    prompt = f"""You are the macro analyst of a team managing an ETF portfolio for the next 2 weeks.

Headlines from the last 7 days, grouped by topic:
{news_text}

Our ETF themes: {", ".join(THEMES)}

Rules:
- Use ONLY the headlines above. Do not invent numbers, dates or facts.
- Ignore headlines that are not relevant for financial markets.
- Our ETFs trade in EUR and many are European: consider the ECB, not only the Fed.
- If topics point in different directions for a theme, give one net view and explain the trade-off.
- Be concise.

For EACH topic write: 1. Key developments (max 3 bullets). 2. Affected ETF themes: theme -> positive/negative, one-line reason. 3. Tone.
Finish with an OVERALL VIEW (2-3 sentences) for the next 2 weeks."""
    for attempt in range(retries + 1):
        try:
            return {"summary": llm.invoke(prompt).content, "headlines": news}
        except Exception as e:
            print(f"AI busy ({type(e).__name__}), waiting {pause}s...", flush=True); time.sleep(pause)
    return {"summary": "[AI unavailable - see raw headlines]", "headlines": news}

## 8. The integrated graph
The **state** is the shared folder passed from node to node:
- `portfolio` current positions · `macro` Gonçalo's summary · `weights` proposed by the Portfolio Manager · `orders` from `weights_to_orders` · `not_executed` moves left for the next day · `risk` Gonçalo's final check

`portfolio_manager_node` is still a **placeholder**: it returns fixed balanced weights. The next step is to replace it with the AI Portfolio Manager (reads `macro`, metrics and signals, returns weights + memo).

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict, total=False):
    portfolio: dict
    macro: dict
    weights: dict
    memo: str
    orders: list
    not_executed: list
    risk: dict

def macro_node(state):
    print("1) Macro/News: headlines + AI summary...", flush=True)
    return {"macro": get_macro_news()}

def portfolio_manager_node(state):
    print("2) Portfolio Manager: proposing target weights (placeholder, no AI yet)...", flush=True)
    weights = {"WORLD": 0.20, "S&P 500": 0.15, "EUROPE 600": 0.10, "GOLD": 0.15,
               "EUR GOVT BOND": 0.15, "DEFENSE": 0.10, "OVERNIGHT CASH": 0.10}
    return {"weights": weights, "memo": "Placeholder: fixed balanced allocation, to be replaced by the AI agent."}

def orders_node(state):
    print("3) Orders: weights -> max 5 feasible orders...", flush=True)
    res = weights_to_orders(state["portfolio"], state["weights"])
    return {"orders": res["orders"], "not_executed": res["not_executed"]}

def risk_node(state):
    print("4) Risk: final check on our risk limits...", flush=True)
    return {"risk": check_risk_rules(state["orders"], to_goncalo_portfolio(state["portfolio"]))}

g = StateGraph(State)
for name, fn in [("macro", macro_node), ("portfolio_manager", portfolio_manager_node),
                 ("orders", orders_node), ("risk", risk_node)]:
    g.add_node(name, fn)
g.add_edge(START, "macro")
g.add_edge("macro", "portfolio_manager")
g.add_edge("portfolio_manager", "orders")
g.add_edge("orders", "risk")
g.add_edge("risk", END)
app = g.compile()
print(app.get_graph().draw_mermaid())

## 9. Run it

In [ ]:
result = app.invoke({"portfolio": new_portfolio()})

print("\n========== MACRO VIEW ==========")
print(result["macro"]["summary"][:3000])
print("\n========== TARGET WEIGHTS ==========")
print(result["weights"], "|", result["memo"])
print("\n========== ORDERS (weights_to_orders) ==========")
display(pd.DataFrame(result["orders"]))
print("Not executed (next trading day):", result["not_executed"])
print("\n========== AFTER RISK CHECK ==========")
for o in result["risk"]["approved"]:
    note = f"  ({o['note']})" if o.get("note") else ""
    print(f"APPROVED  {o['action'].upper():4} {o['code']:<15} {o['amount_eur']:>9,.2f} EUR{note}")
for o in result["risk"]["rejected"]:
    print(f"REJECTED  {o['action'].upper():4} {o['code']:<15} -> {o['reason']}")
print(f"\nCash after: {result['risk']['cash_after_eur']:,.2f} EUR")
print("Weights after:", result["risk"]["weights_after"])